# Basics

The same notebook as [`../basics.ipynb`](../basics.ipynb), written against
[`ib_async`](https://github.com/ib-api-reloaded/ib_async) instead of the TWS API
shape. The library is unmodified and installed as usual; `ibx.ib_async.attach`
replaces the one layer of it that expects a socket to a gateway.

Account values, positions, and one quote.

## Connecting

`IB.connect` was written for a gateway, so it takes a host, a port and a client
id. Here it takes none of them: the credentials go to `attach`, and there is no
local process to reach.

`ib.sleep()` rather than `time.sleep()` throughout. The library's loop runs on
this thread, and a plain sleep stops it — every stream then reads as dead.

In [ ]:
import os
from dotenv import load_dotenv
from ib_async import IB, util
import ibx.ib_async

util.startLoop()
load_dotenv()

ib = ibx.ib_async.attach(
    IB(),
    username=os.environ["IB_USERNAME"],
    password=os.environ["IB_PASSWORD"],
    paper=True,
)
ib.connect()          # names no host: there is no gateway to name

print(f"connected: {ib.isConnected()}")
print(f"accounts:  {ib.managedAccounts()}")

## The account

What this login holds, and what it is worth.

In [ ]:
summary = ib.accountSummary()
for row in summary:
    if row.tag in ("NetLiquidation", "AvailableFunds", "BuyingPower"):
        print(f"{row.tag:20} {row.value:>16} {row.currency}")

## Positions

Answered from what the session has already been told, so this returns at once.

In [ ]:
positions = ib.positions()
for p in positions:
    print(f"{p.contract.symbol:8} {p.position:>10}  avg {p.avgCost:.2f}")
if not positions:
    print("no open positions")

## A quote

Qualify the contract first: the venue answers a request that names a
contract completely, and `qualifyContracts` is what fills in the rest.

In [ ]:
from ib_async import Stock

spy = Stock("SPY", "SMART", "USD")
ib.qualifyContracts(spy)
print(spy)

ticker = ib.reqMktData(spy)
ib.sleep(3)
print(f"bid {ticker.bid}  ask {ticker.ask}  last {ticker.last}")
ib.cancelMktData(spy)

## Disconnecting

In [ ]:
ib.disconnect()
print(f"connected: {ib.isConnected()}")